# Refugio Animal: gestor de animales con Tkinter

## Primera versión del proyecto

Este cuaderno toma como punto de partida la temática y los datos comunes del proyecto `poo_animales` (perros, gatos y loros; nombre, edad, peso y color) y los convierte en una pequeña aplicación de escritorio.

**Objetivo:** practicar diccionarios, funciones, condicionales, excepciones y controles de Tkinter construyendo las cuatro operaciones CRUD:

| Operación | Acción en la aplicación |
| --- | --- |
| Crear | Añadir un animal al inventario |
| Leer | Mostrar, buscar y seleccionar registros |
| Actualizar | Modificar los datos del animal seleccionado |
| Borrar | Eliminar el registro seleccionado |

Los datos viven en un diccionario de Python en memoria; no se utiliza una base de datos. Por ello, al reiniciar el kernel se recuperarán los datos de ejemplo y se perderán los cambios de la sesión.

## Documentación de `Tema3-InterfacesGraficas` utilizada

La interfaz se ha planteado siguiendo los controles y ejemplos de estas lecciones del curso. Los enlaces son relativos a la ubicación de este cuaderno:

| Lección | Aplicación en el proyecto |
| --- | --- |
| [02 - Gestores de geometría](../../../Tema3-InterfacesGraficas/02_GestoresGeometria.ipynb) | `pack()` para los bloques principales y `grid()` en contenedores internos, sin mezclarlos dentro del mismo contenedor. |
| [03 - Variables de control](../../../Tema3-InterfacesGraficas/03_VariablesControl.ipynb) | `StringVar`, `get()` y `set()` para enlazar los campos del formulario. |
| [04 - Frame y LabelFrame](../../../Tema3-InterfacesGraficas/04_Contenedor_Frame_Labelframe.ipynb) | Agrupar los datos, los filtros y los controles de la tabla. |
| [08 - Button](../../../Tema3-InterfacesGraficas/08_Widget_Button.ipynb) y [09 - Entry](../../../Tema3-InterfacesGraficas/09_Widget_Entry.ipynb) | Botones con `command` y campos de entrada enlazados a variables. |
| [10 - Eventos](../../../Tema3-InterfacesGraficas/10_EventosGenericos.ipynb) | `bind()` para actualizar la búsqueda al escribir y responder a selecciones. |
| [11 - Cajas de diálogo](../../../Tema3-InterfacesGraficas/11_CajasDialogo.ipynb) | Avisos, errores y confirmación de borrado con `messagebox`. |
| [13 - Widgets temáticos TTK](../../../Tema3-InterfacesGraficas/13_WidgetsTematicos_TTK.ipynb) | Uso de controles `ttk` para mantener una apariencia uniforme. |
| [17 - Combobox](../../../Tema3-InterfacesGraficas/17_Widget_Combobox.ipynb) y [18 - Text](../../../Tema3-InterfacesGraficas/18_Widget_Text.ipynb) | Selección de especie y edición de observaciones multilínea. |
| [22 - Treeview](../../../Tema3-InterfacesGraficas/22_Widget_Treeview.ipynb) | Tabla, columnas, selección de una fila y evento `<<TreeviewSelect>>`. |

Conviene abrir estas lecciones junto al cuaderno para consultar la explicación de cada control y de sus métodos.

## 1. Datos iniciales y estructura

`animales` es un diccionario cuyas claves son identificadores numéricos únicos. Cada valor es otro diccionario con los datos de un animal. Esta estructura permite localizar y modificar un registro a partir de su identificador.

Se incluyen varios campos para que la gestión sea algo más completa que una lista simple: especie, edad, peso, color y observaciones. La edad admite valores de 0 a 30, como en el modelo del proyecto de referencia.

In [6]:
import math
import tkinter as tk
from tkinter import messagebox, ttk


ESPECIES = ("Perro", "Gato", "Loro")

animales = {
    1: {
        "nombre": "Luna",
        "especie": "Perro",
        "edad": 4,
        "peso": 18.5,
        "color": "Canela",
        "observaciones": "Sociable; disfruta de los paseos.",
    },
    2: {
        "nombre": "Misi",
        "especie": "Gato",
        "edad": 3,
        "peso": 4.2,
        "color": "Blanco y gris",
        "observaciones": "Tranquilo; busca una familia paciente.",
    },
    3: {
        "nombre": "Kiko",
        "especie": "Loro",
        "edad": 7,
        "peso": 0.6,
        "color": "Verde",
        "observaciones": "Le gusta escuchar música.",
    },
    4: {
        "nombre": "Toby",
        "especie": "Perro",
        "edad": 1,
        "peso": 8.3,
        "color": "Negro",
        "observaciones": "Juguetón y con mucha energía.",
    },
}

print(f"Registros iniciales: {len(animales)}")
print("Primer registro:", animales[1])

Registros iniciales: 4
Primer registro: {'nombre': 'Luna', 'especie': 'Perro', 'edad': 4, 'peso': 18.5, 'color': 'Canela', 'observaciones': 'Sociable; disfruta de los paseos.'}


## 2. Operaciones CRUD y validación

Las funciones siguientes separan la gestión de datos de la interfaz. Reciben el diccionario como argumento, lo que permite probarlas con un inventario temporal. Los errores de validación se comunican mediante `ValueError`; más adelante la interfaz los mostrará con un cuadro de diálogo.

In [7]:
def validar_datos(datos):
    """Comprueba los datos del formulario y devuelve un registro limpio."""
    nombre = datos.get("nombre", "").strip()
    especie = datos.get("especie", "").strip()
    color = datos.get("color", "").strip()
    observaciones = datos.get("observaciones", "").strip()
    edad = datos.get("edad")

    if not nombre:
        raise ValueError("El nombre es obligatorio.")
    if especie not in ESPECIES:
        raise ValueError("Selecciona una especie de la lista.")
    if not color:
        raise ValueError("El color es obligatorio.")
    if isinstance(edad, bool) or not isinstance(edad, int):
        raise ValueError("La edad debe ser un número entero.")
    if edad < 0 or edad > 30:
        raise ValueError("La edad debe estar entre 0 y 30 años.")

    try:
        peso = float(datos.get("peso"))
    except (TypeError, ValueError) as error:
        raise ValueError("El peso debe ser un número válido.") from error
    if not math.isfinite(peso) or peso <= 0:
        raise ValueError("El peso debe ser un número finito mayor que cero.")

    return {
        "nombre": nombre,
        "especie": especie,
        "edad": edad,
        "peso": peso,
        "color": color,
        "observaciones": observaciones,
    }


def crear_animal(datos, inventario):
    """Valida y añade un animal; devuelve el nuevo identificador."""
    registro = validar_datos(datos)
    nuevo_id = max(inventario, default=0) + 1
    inventario[nuevo_id] = registro
    return nuevo_id


def leer_animal(animal_id, inventario):
    """Devuelve una copia de un registro para evitar cambios accidentales."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    return inventario[animal_id].copy()


def actualizar_animal(animal_id, datos, inventario):
    """Valida y sustituye los datos del animal indicado."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    inventario[animal_id] = validar_datos(datos)


def borrar_animal(animal_id, inventario):
    """Elimina el animal indicado y devuelve sus datos."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    return inventario.pop(animal_id)


## 3. Prueba rápida de las cuatro operaciones

Esta celda usa un diccionario vacío independiente para no cambiar los registros iniciales. En la salida se puede observar cómo cambia el inventario al crear, leer, actualizar y borrar.

In [8]:
inventario_prueba = {}
datos_prueba = {
    "nombre": "Nube",
    "especie": "Gato",
    "edad": 2,
    "peso": 3.8,
    "color": "Gris",
    "observaciones": "Ejemplo temporal",
}

id_prueba = crear_animal(datos_prueba, inventario_prueba)  # Crear
print("Crear:", id_prueba, inventario_prueba)
print("Leer:", leer_animal(id_prueba, inventario_prueba))

datos_prueba["edad"] = 3
actualizar_animal(id_prueba, datos_prueba, inventario_prueba)  # Actualizar
print("Actualizar:", leer_animal(id_prueba, inventario_prueba))

borrar_animal(id_prueba, inventario_prueba)  # Borrar
print("Borrar: quedan", len(inventario_prueba), "registros.")

Crear: 1 {1: {'nombre': 'Nube', 'especie': 'Gato', 'edad': 2, 'peso': 3.8, 'color': 'Gris', 'observaciones': 'Ejemplo temporal'}}
Leer: {'nombre': 'Nube', 'especie': 'Gato', 'edad': 2, 'peso': 3.8, 'color': 'Gris', 'observaciones': 'Ejemplo temporal'}
Actualizar: {'nombre': 'Nube', 'especie': 'Gato', 'edad': 3, 'peso': 3.8, 'color': 'Gris', 'observaciones': 'Ejemplo temporal'}
Borrar: quedan 0 registros.


## 4. Interfaz gráfica

La ventana utiliza controles estándar de Tkinter: `LabelFrame`, `Label`, `Entry`, `Combobox`, `Text`, `Button` y `Treeview`. Se puede escribir parte de un nombre para buscarlo y limitar la tabla a una especie. Al seleccionar una fila, sus datos pasan al formulario para poder actualizarlos o borrarlos.

In [9]:
class GestorAnimales:
    """Interfaz Tkinter para consultar y gestionar el inventario."""

    def __init__(self, raiz, inventario):
        self.raiz = raiz
        self.inventario = inventario
        self.nombre_var = tk.StringVar()
        self.especie_var = tk.StringVar(value=ESPECIES[0])
        self.edad_var = tk.StringVar()
        self.peso_var = tk.StringVar()
        self.color_var = tk.StringVar()
        self.busqueda_var = tk.StringVar()
        self.filtro_var = tk.StringVar(value="Todas")
        self.estado_var = tk.StringVar(value="Selecciona un animal o completa el formulario.")

        self.raiz.title("Refugio Animal - Gestion de animales")
        self.raiz.geometry("1050x720")
        self.raiz.minsize(850, 620)
        self._crear_controles()
        self._actualizar_tabla()

    def _crear_controles(self):
        """Construye el formulario, los filtros y la tabla."""
        contenedor = ttk.Frame(self.raiz, padding=12)
        contenedor.pack(fill="both", expand=True)

        ttk.Label(contenedor, text="Refugio Animal").pack(anchor="w")
        ttk.Label(
            contenedor,
            text="Registro de perros, gatos y loros",
        ).pack(anchor="w", pady=(0, 10))

        formulario = ttk.LabelFrame(contenedor, text="Datos del animal", padding=10)
        formulario.pack(fill="x", pady=(0, 10))
        formulario.columnconfigure(1, weight=1)
        formulario.columnconfigure(3, weight=1)

        ttk.Label(formulario, text="Nombre:").grid(row=0, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.nombre_var).grid(
            row=0, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Especie:").grid(row=0, column=2, sticky="w", padx=5, pady=4)
        ttk.Combobox(
            formulario,
            textvariable=self.especie_var,
            values=ESPECIES,
            state="readonly",
        ).grid(row=0, column=3, sticky="ew", padx=5, pady=4)

        ttk.Label(formulario, text="Edad (0-30):").grid(row=1, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.edad_var).grid(
            row=1, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Peso (kg):").grid(row=1, column=2, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.peso_var).grid(
            row=1, column=3, sticky="ew", padx=5, pady=4
        )

        ttk.Label(formulario, text="Color:").grid(row=2, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.color_var).grid(
            row=2, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Observaciones:").grid(
            row=2, column=2, sticky="nw", padx=5, pady=4
        )
        self.observaciones_text = tk.Text(formulario, height=3, width=35, wrap="word")
        self.observaciones_text.grid(row=2, column=3, sticky="ew", padx=5, pady=4)

        botones = ttk.Frame(formulario)
        botones.grid(row=3, column=0, columnspan=4, sticky="w", pady=(8, 0))
        ttk.Button(botones, text="Crear", command=self._crear).pack(side="left", padx=3)
        ttk.Button(botones, text="Actualizar", command=self._actualizar).pack(side="left", padx=3)
        ttk.Button(botones, text="Borrar", command=self._borrar).pack(side="left", padx=3)
        ttk.Button(botones, text="Limpiar", command=self._limpiar).pack(side="left", padx=3)

        filtros = ttk.LabelFrame(contenedor, text="Buscar y filtrar", padding=8)
        filtros.pack(fill="x", pady=(0, 8))
        ttk.Label(filtros, text="Nombre:").pack(side="left", padx=(0, 5))
        busqueda = ttk.Entry(filtros, textvariable=self.busqueda_var, width=28)
        busqueda.pack(side="left", padx=(0, 14))
        busqueda.bind("<KeyRelease>", lambda evento: self._actualizar_tabla())
        ttk.Label(filtros, text="Especie:").pack(side="left", padx=(0, 5))
        filtro = ttk.Combobox(
            filtros,
            textvariable=self.filtro_var,
            values=("Todas",) + ESPECIES,
            state="readonly",
            width=14,
        )
        filtro.pack(side="left")
        filtro.bind("<<ComboboxSelected>>", lambda evento: self._actualizar_tabla())

        tabla_marco = ttk.Frame(contenedor)
        tabla_marco.pack(fill="both", expand=True)
        columnas = ("id", "nombre", "especie", "edad", "peso", "color", "observaciones")
        self.tabla = ttk.Treeview(
            tabla_marco,
            columns=columnas,
            show="headings",
            selectmode="browse",
            height=10,
        )
        encabezados = {
            "id": "ID",
            "nombre": "Nombre",
            "especie": "Especie",
            "edad": "Edad",
            "peso": "Peso (kg)",
            "color": "Color",
            "observaciones": "Observaciones",
        }
        anchos = {"id": 45, "nombre": 110, "especie": 85, "edad": 55,
                  "peso": 75, "color": 100, "observaciones": 300}
        for columna in columnas:
            self.tabla.heading(columna, text=encabezados[columna])
            self.tabla.column(columna, width=anchos[columna], anchor="w")
        barra_vertical = ttk.Scrollbar(tabla_marco, orient="vertical", command=self.tabla.yview)
        barra_horizontal = ttk.Scrollbar(tabla_marco, orient="horizontal", command=self.tabla.xview)
        self.tabla.configure(
            yscrollcommand=barra_vertical.set,
            xscrollcommand=barra_horizontal.set,
        )
        self.tabla.grid(row=0, column=0, sticky="nsew")
        barra_vertical.grid(row=0, column=1, sticky="ns")
        barra_horizontal.grid(row=1, column=0, sticky="ew")
        tabla_marco.rowconfigure(0, weight=1)
        tabla_marco.columnconfigure(0, weight=1)
        self.tabla.bind("<<TreeviewSelect>>", self._seleccionar)

        ttk.Label(contenedor, textvariable=self.estado_var).pack(anchor="w", pady=(8, 0))

    def _datos_del_formulario(self):
        """Lee los controles y convierte edad y peso a valores numéricos."""
        try:
            edad = int(self.edad_var.get().strip())
        except ValueError as error:
            raise ValueError("La edad debe escribirse como un número entero.") from error
        try:
            peso = float(self.peso_var.get().strip())
        except ValueError as error:
            raise ValueError("El peso debe escribirse como un número, por ejemplo 4.5.") from error

        return {
            "nombre": self.nombre_var.get(),
            "especie": self.especie_var.get(),
            "edad": edad,
            "peso": peso,
            "color": self.color_var.get(),
            "observaciones": self.observaciones_text.get("1.0", "end"),
        }

    def _id_seleccionado(self):
        """Devuelve el ID de la fila seleccionada o muestra una advertencia."""
        seleccion = self.tabla.selection()
        if not seleccion:
            messagebox.showwarning("Falta seleccionar", "Selecciona un animal de la tabla.")
            return None
        return int(seleccion[0])

    def _crear(self):
        try:
            animal_id = crear_animal(self._datos_del_formulario(), self.inventario)
        except ValueError as error:
            messagebox.showerror("No se pudo crear el registro", str(error))
            return
        self._actualizar_tabla()
        self._limpiar()
        self.estado_var.set(f"Animal creado con el identificador {animal_id}.")

    def _actualizar(self):
        animal_id = self._id_seleccionado()
        if animal_id is None:
            return
        try:
            actualizar_animal(animal_id, self._datos_del_formulario(), self.inventario)
        except ValueError as error:
            messagebox.showerror("No se pudo actualizar el registro", str(error))
            return
        self._actualizar_tabla()
        self.estado_var.set(f"Registro {animal_id} actualizado.")

    def _borrar(self):
        animal_id = self._id_seleccionado()
        if animal_id is None:
            return
        nombre = self.inventario[animal_id]["nombre"]
        if not messagebox.askyesno("Confirmar borrado", f"¿Quieres borrar a {nombre}?"):
            return
        try:
            borrar_animal(animal_id, self.inventario)
        except ValueError as error:
            messagebox.showerror("No se pudo borrar el registro", str(error))
            return
        self._actualizar_tabla()
        self._limpiar()
        self.estado_var.set(f"Registro {animal_id} eliminado.")

    def _seleccionar(self, evento=None):
        seleccion = self.tabla.selection()
        if not seleccion:
            return
        animal_id = int(seleccion[0])
        registro = leer_animal(animal_id, self.inventario)
        self.nombre_var.set(registro["nombre"])
        self.especie_var.set(registro["especie"])
        self.edad_var.set(str(registro["edad"]))
        self.peso_var.set(str(registro["peso"]))
        self.color_var.set(registro["color"])
        self.observaciones_text.delete("1.0", "end")
        self.observaciones_text.insert("1.0", registro["observaciones"])
        self.estado_var.set(f"Registro {animal_id} seleccionado.")

    def _limpiar(self):
        self.nombre_var.set("")
        self.especie_var.set(ESPECIES[0])
        self.edad_var.set("")
        self.peso_var.set("")
        self.color_var.set("")
        self.observaciones_text.delete("1.0", "end")
        seleccion = self.tabla.selection()
        if seleccion:
            self.tabla.selection_remove(seleccion)

    def _actualizar_tabla(self):
        """Vuelve a dibujar los registros aplicando búsqueda y filtro."""
        for fila in self.tabla.get_children():
            self.tabla.delete(fila)

        texto = self.busqueda_var.get().strip().casefold()
        especie = self.filtro_var.get()
        visibles = 0
        for animal_id, registro in self.inventario.items():
            coincide_nombre = texto in registro["nombre"].casefold()
            coincide_especie = especie == "Todas" or registro["especie"] == especie
            if coincide_nombre and coincide_especie:
                self.tabla.insert(
                    "",
                    "end",
                    iid=str(animal_id),
                    values=(
                        animal_id,
                        registro["nombre"],
                        registro["especie"],
                        registro["edad"],
                        f"{registro['peso']:.2f}",
                        registro["color"],
                        registro["observaciones"],
                    ),
                )
                visibles += 1
        self.estado_var.set(
            f"Registros totales: {len(self.inventario)} | visibles: {visibles}"
        )

## 5. Ejecutar la aplicación

Ejecuta esta última celda para abrir la ventana. La llamada a `mainloop()` mantiene la interfaz activa hasta que se cierre la ventana. Si el entorno donde se abre el notebook no dispone de escritorio gráfico, ejecuta el cuaderno localmente en VS Code o adapta la clase para una aplicación web.

Orden recomendado: **Reiniciar kernel y ejecutar todas las celdas**. La ventana se crea al ejecutar esta celda, no al importar o definir las funciones.

In [10]:
raiz = tk.Tk()
aplicacion = GestorAnimales(raiz, animales)
raiz.mainloop()